In [1]:
%cd ~/
!rm -rf SMILES-2026-Hallucination-Detection
!git clone https://github.com/p4sttt/SMILES-2026-Hallucination-Detection.git
%cd SMILES-2026-Hallucination-Detection
!ls -ls

/root
Cloning into 'SMILES-2026-Hallucination-Detection'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 64 (delta 29), reused 32 (delta 17), pack-reused 12 (from 1)
Receiving objects: 100% (64/64), 466.52 KiB | 2.71 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/root/SMILES-2026-Hallucination-Detection
total 108
12 -rw-r--r-- 1 root root  9225 May  8 20:56 aggregation.py
 4 drwxr-xr-x 2 root root  4096 May  8 20:56 data
16 -rw-r--r-- 1 root root 12572 May  8 20:56 evaluate.py
 4 -rw-r--r-- 1 root root  1063 May  8 20:56 LICENSE
 4 -rw-r--r-- 1 root root   113 May  8 20:56 main.py
 4 -rw-r--r-- 1 root root  1381 May  8 20:56 model.py
 8 -rw-r--r-- 1 root root  4883 May  8 20:56 probe.py
 4 -rw-r--r-- 1 root root   186 May  8 20:56 pyproject.toml
 8 -rw-r--r-- 1 root root  5713 May  8 20:56 README.md
 4 -rw-r--r-- 1 root root   112 May  8 20:56 requirements.txt
24 -rw-r--r-- 1 roo

# Classifier Experiments

Отдельный notebook для сравнения альтернативных probe-классификаторов без смешивания с финальным `solution.ipynb` / `probe.py`.

Использует те же признаки, `split_data`, `run_evaluation`, `print_summary` и `save_results`, что финальное решение.

## 1. Setup

В Colab включи GPU runtime. После установки зависимостей, если Colab просит restart runtime или ломается импорт `pandas`, перезапусти runtime и начни с ячейки imports.

In [2]:
from pathlib import Path

if not Path("requirements.txt").exists():
    raise FileNotFoundError("Run this notebook from the repository root.")

!pip install -q -r requirements.txt optuna xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 2.5 MB/s eta 0:00:00a 0:00:01


## 2. Imports And Config

In [3]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from xgboost import XGBClassifier

from aggregation import aggregation_and_feature_extraction
from evaluate import print_summary, run_evaluation, save_results
from model import MAX_LENGTH, get_model_and_tokenizer
from splitting import split_data

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_FILE = "./data/dataset.csv"
BATCH_SIZE = 4
USE_GEOMETRIC = False
FEATURE_CACHE = "experiment_features.npz"
RESULTS_DIR = Path("experiment_results")
RESULTS_DIR.mkdir(exist_ok=True)

# Increase for slower but deeper searches. 20-40 is usually enough for a first pass.
N_TRIALS = 25
RANDOM_STATE = 42

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device        : {device}")
print(f"Max length    : {MAX_LENGTH}")
print(f"Geometric feats: {USE_GEOMETRIC}")
print(f"Optuna trials : {N_TRIALS}")

Device        : cpu
Max length    : 512
Geometric feats: False
Optuna trials : 25


## 3. Feature Extraction

Признаки кэшируются в `experiment_features.npz`, чтобы не гонять Qwen заново для каждого эксперимента.

In [5]:
df = pd.read_csv(DATA_FILE)
all_texts = [f"{row['prompt']}{row['response']}" for _, row in df.iterrows()]
y = np.array([int(float(label)) for label in df["label"]])

print(f"Loaded {len(y)} samples")
print(f"Labels: {dict(pd.Series(y).value_counts().sort_index())}")

Loaded 689 samples
Labels: {0: np.int64(206), 1: np.int64(483)}


In [6]:
def extract_features(texts: list[str]) -> tuple[np.ndarray, float]:
    model, tokenizer = get_model_and_tokenizer()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.to(device)
    model.eval()

    features = []
    t0 = time.time()

    for start in tqdm(
        range(0, len(texts), BATCH_SIZE), desc="Extracting", unit="batch"
    ):
        batch_texts = texts[start : start + BATCH_SIZE]
        encoding = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        hidden = torch.stack(outputs.hidden_states, dim=1).float()
        mask = attention_mask.cpu()

        for i in range(hidden.size(0)):
            feat = aggregation_and_feature_extraction(
                hidden[i],
                mask[i],
                use_geometric=USE_GEOMETRIC,
            )
            features.append(feat.cpu())

        del input_ids, attention_mask, outputs, hidden, mask
        if device.type == "cuda":
            torch.cuda.empty_cache()

    elapsed = time.time() - t0
    X = np.vstack([f.numpy() for f in features])
    return X, elapsed

In [7]:
if Path(FEATURE_CACHE).exists():
    cached = np.load(FEATURE_CACHE)
    X = cached["X"]
    y = cached["y"]
    extract_time = float(cached["extract_time"])
    print(f"Loaded cached features from {FEATURE_CACHE}")
else:
    X, extract_time = extract_features(all_texts)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y, extract_time=extract_time)
    print(f"Saved cached features to {FEATURE_CACHE}")

print(f"Feature matrix: {X.shape}")
print(f"Extract time  : {extract_time:.1f} s")

[Model] Loading 'Qwen/Qwen2.5-0.5B' ...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Extracting:   0%|          | 0/173 [00:00<?, ?batch/s]

KeyboardInterrupt: 

## 4. Shared Probe Helpers

Все классы ниже реализуют тот же API, что `HallucinationProbe`: `fit`, `fit_hyperparameters`, `predict`, `predict_proba`.

In [ ]:
def best_accuracy_threshold(y_true: np.ndarray, probs: np.ndarray) -> tuple[float, float]:
    sorted_probs = np.unique(probs)
    midpoints = (sorted_probs[:-1] + sorted_probs[1:]) / 2.0
    candidates = np.unique(np.concatenate([[0.0, 1.0], sorted_probs, midpoints]))

    best_t = 0.5
    best_acc = -1.0
    for t in candidates:
        acc = accuracy_score(y_true, (probs >= t).astype(int))
        tie_break = acc == best_acc and abs(t - 0.5) < abs(best_t - 0.5)
        if acc > best_acc or tie_break:
            best_acc = acc
            best_t = float(t)
    return best_t, float(best_acc)


class BaseOptunaProbe:
    name = "base"

    def __init__(self) -> None:
        self._clf = None
        self._threshold = 0.5
        self._X_train = None
        self._y_train = None
        self._best_params = None

    def _default_clf(self):
        raise NotImplementedError

    def _suggest_clf(self, trial: optuna.Trial):
        raise NotImplementedError

    def fit(self, X: np.ndarray, y: np.ndarray):
        self._X_train = X.copy()
        self._y_train = y.copy()
        self._clf = self._default_clf()
        self._clf.fit(X, y)
        self._threshold = 0.5
        return self

    def fit_hyperparameters(self, X_val: np.ndarray, y_val: np.ndarray):
        if self._X_train is None or self._y_train is None:
            raise RuntimeError("Call fit() before fit_hyperparameters().")

        best_threshold = 0.5

        def objective(trial: optuna.Trial) -> float:
            nonlocal best_threshold
            clf = self._suggest_clf(trial)
            clf.fit(self._X_train, self._y_train)
            probs = clf.predict_proba(X_val)[:, 1]
            threshold, acc = best_accuracy_threshold(y_val, probs)
            try:
                auroc = roc_auc_score(y_val, probs)
            except ValueError:
                auroc = 0.0
            trial.set_user_attr("threshold", threshold)
            trial.set_user_attr("auroc", float(auroc))
            return acc

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

        self._best_params = study.best_params
        self._threshold = float(study.best_trial.user_attrs.get("threshold", best_threshold))
        self._clf = self._suggest_clf(study.best_trial)
        self._clf.fit(self._X_train, self._y_train)

        print(
            f"{self.name}: best val accuracy={study.best_value:.4f}, "
            f"threshold={self._threshold:.4f}, params={self._best_params}"
        )
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self.predict_proba(X)[:, 1] >= self._threshold).astype(int)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if self._clf is None:
            raise RuntimeError("Probe is not fitted.")
        return self._clf.predict_proba(X)

## 5. Alternative Classifiers

In [ ]:
class RandomForestOptunaProbe(BaseOptunaProbe):
    name = "random_forest"

    def _default_clf(self):
        return RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight=None,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

    def _suggest_clf(self, trial: optuna.Trial):
        return RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 700, step=100),
            max_depth=trial.suggest_categorical("max_depth", [None, 3, 5, 7, 10, 14]),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 12),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )


class XGBoostOptunaProbe(BaseOptunaProbe):
    name = "xgboost"

    def _default_clf(self):
        return XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            objective="binary:logistic",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

    def _suggest_clf(self, trial: optuna.Trial):
        n_pos = max(int(self._y_train.sum()), 1)
        n_neg = max(len(self._y_train) - n_pos, 1)
        return XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 50, 500, step=50),
            max_depth=trial.suggest_int("max_depth", 1, 5),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            subsample=trial.suggest_float("subsample", 0.55, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.55, 1.0),
            min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            gamma=trial.suggest_float("gamma", 0.0, 5.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 30.0, log=True),
            scale_pos_weight=trial.suggest_categorical("scale_pos_weight", [1.0, n_neg / n_pos]),
            eval_metric="logloss",
            objective="binary:logistic",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )


class MLPOptunaProbe(BaseOptunaProbe):
    name = "mlp"

    def _default_clf(self):
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "mlp",
                    MLPClassifier(
                        hidden_layer_sizes=(32,),
                        alpha=1e-3,
                        learning_rate_init=1e-3,
                        early_stopping=True,
                        max_iter=800,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )

    def _suggest_clf(self, trial: optuna.Trial):
        hidden_layer_sizes = trial.suggest_categorical(
            "hidden_layer_sizes",
            [(16,), (32,), (64,), (32, 16), (64, 32)],
        )
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "mlp",
                    MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
                        alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                        learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 5e-2, log=True),
                        batch_size=trial.suggest_categorical("batch_size", [16, 32, 64]),
                        early_stopping=True,
                        validation_fraction=0.15,
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )

## 6. Run Evaluation

Эта ячейка использует тот же `split_data` и тот же `run_evaluation`, что финальный запуск. Каждый результат сохраняется отдельным JSON в `experiment_results/`.

In [ ]:
splits = split_data(y, df)

print(f"Splits: {len(splits)} fold(s)")
for i, (tr, va, te) in enumerate(splits, 1):
    print(f"  Fold {i}: train={len(tr)} val={len(va) if va is not None else 'N/A'} test={len(te)}")

MODEL_REGISTRY = {
    "random_forest": RandomForestOptunaProbe,
    "xgboost": XGBoostOptunaProbe,
    "mlp": MLPOptunaProbe,
}

experiment_summaries = []

for model_name, ProbeClass in MODEL_REGISTRY.items():
    print("\n" + "#" * 80)
    print(f"Running {model_name}")
    print("#" * 80)

    fold_results = run_evaluation(splits, X, y, ProbeClass)
    print_summary(fold_results, X.shape[1], len(X), extract_time)

    output_file = RESULTS_DIR / f"{model_name}.json"
    save_results(fold_results, X.shape[1], len(X), extract_time, str(output_file))

    with open(output_file) as f:
        summary = json.load(f)
    summary["model"] = model_name
    experiment_summaries.append(summary)

Splits: 5 fold(s)
  Fold 1: train=468 val=83 test=138
  Fold 2: train=468 val=83 test=138
  Fold 3: train=468 val=83 test=138
  Fold 4: train=468 val=83 test=138
  Fold 5: train=469 val=83 test=137

################################################################################
Running random_forest
################################################################################

──────────────────────────────────────────────────
  Fold 1/5  —  train=468  val=83  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
random_forest: best val accuracy=0.7470, threshold=0.5679, params={'n_estimators': 600, 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 3, 'max_features': 'sqrt', 'class_weight': None, 'bootstrap': True}
  Probe train — Acc: 100.00%  F1: 100.00%  AUROC: 100.00%
  Probe val  — Acc: 74.70%  F1: 82.05%  AUROC: 72.69%
  Probe test — Acc: 72.46%  F1: 81.73%  AUROC: 70.28%

─────────────────────────────────────────────────

## 7. Compare Results

In [ ]:
def nanmean_from_folds(summary: dict, key: str) -> float:
    values = [fold.get(key, np.nan) for fold in summary["folds"]]
    return float(np.nanmean(values))


comparison = pd.DataFrame(
    [
        {
            "model": s["model"],
            "baseline_acc": s["avg_baseline_accuracy"],
            "train_acc": s["avg_train_accuracy"],
            "val_acc": nanmean_from_folds(s, "val_accuracy"),
            "test_acc": s["avg_test_accuracy"],
            "test_f1": s["avg_test_f1"],
            "test_auroc": s["avg_test_auroc"],
        }
        for s in experiment_summaries
    ]
).sort_values("test_acc", ascending=False)

comparison.to_csv(RESULTS_DIR / "comparison.csv", index=False)
comparison

,model,baseline_acc,train_acc,val_acc,test_acc,test_f1,test_auroc
1,xgboost,0.701016,0.880431,0.775904,0.724225,0.822688,0.691047
0,random_forest,0.701016,0.901751,0.754217,0.722808,0.825674,0.706026
2,mlp,0.701016,0.797098,0.768675,0.692394,0.796849,0.668101
